### Will, precisamos iniciar a modelagem dos dados dos nossos clientes, mas a equipe de visualização não pode acessar o arquivo cru no Data Lake.
### Preciso que você crie um banco de dados focado em CRM e ingira aquele arquivo CSV de Clientes como a nossa tabela bruta oficial (raw) para que possamos plugar o Power BI.



**Exercicio feito apenas para praticar modelagem de dados**

In [0]:
%fs
ls dbfs:/

path,name,size,modificationTime
dbfs:/Volumes/,Volumes/,0,0
dbfs:/Workspace/,Workspace/,0,0
dbfs:/databricks-datasets/,databricks-datasets/,0,0


In [0]:
%fs
ls '/Volumes/workspace/default/arq-aula-dbs/'

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/arq-aula-dbs/Anac/,Anac/,0,1781101577068
dbfs:/Volumes/workspace/default/arq-aula-dbs/Bike Store/,Bike Store/,0,1781101577069
dbfs:/Volumes/workspace/default/arq-aula-dbs/arquivos_csv/,arquivos_csv/,0,1781101577069


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS WORKSPACE.CRM_ANALYTICS

- criar um db focado em CRM (CRM_ANALYTICS).
Criar uma tabela chamada clientes_raw dentro do Db (CRM_ANALYTICS).

In [0]:
file_path = ('dbfs:/Volumes/workspace/default/arq-aula-dbs/arquivos_csv/Clientes.csv')
 
df_cliente_raw = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ",")
    .load(file_path))
df_cliente_raw.write.mode('overwrite').saveAsTable('WORKSPACE.CRM_ANALYTICS.clientesRaw')


In [0]:
%sql
-- Dados que serao tratados, alterando os valores de null

select first_name, last_name, email, street, number, additionals 
 from workspace.crm_analytics.clientesraw
 limit 10

first_name,last_name,email,street,number,additionals
Marta,Jesus,null,null,null,Conjunto 16
Luana,Almeida,null,Avenida 56 do Estado Rio Grande do Sul,989.0,Conjunto 17
Frida,Mendes,frida@meu_email.com,Avenida 59 do Estado São Paulo,534.0,null
Daniela,Avelino,daniela@exemplo.com,null,null,null
Romário,Teixeira,null,Praça 56 do Estado Bahia,191.0,Apto 12
Marcelo,Barroso,null,Rua 28 do Estado Rio Grande do Sul,805.0,Conjunto 13
Cristiano,Elísio,cristiano@exemplo.com,Rua 78 do Estado Goiás,877.0,Apto 14
Everton,Barbosa,everton@meu_email.com,Avenida 86 do Estado Distrito Federal,864.0,Apto 14
Gabriela,Alves,gabriela@exemplo.com,null,null,null
Luan,Dias,luan@exemplo.com,Avenida 54 do Estado Distrito Federal,889.0,Conjunto 14


**Will, excelente trabalho na ingestão da base bruta. Agora precisamos de uma versão limpa dessa tabela para plugar nas nossas ferramentas operacionais.
O nosso sistema de disparo de marketing quebra se receber um campo de email como 'null'. Além disso, o sistema de roteirização da logística não aceita 'null' na coluna de número da casa, ele precisa de um número real.**

**1.Alterar os valores de email de nulo para nao informado.**

**2.Alterar os valores de casa(additionals) null para 0.**

 **3. Salvar df como uma nova tabela chamada cliente_silver, dentro do schema.workspace.CRM_ANALYTICS.**



In [0]:
%sql
CREATE TABLE IF NOT EXISTS WORKSPACE.crm_analytics.cliente_silver

In [0]:
df_clientes_silver = (spark.read.table('workspace.crm_analytics.clientesraw')
    .na.fill({'state':'estado_nao_informado', 
              'email':'nao_informado', 
              'street':'nao_informado', 
              'number': 0, 
              'additionals': 'sem_complemento' }))
df_clientes_silver.write.mode('overwrite').saveAsTable('workspace.crm_analytics.cliente_silver')


In [0]:
%sql
-- Mostrando apenas os dados que foram tratados.

select first_name, last_name, email, street, number, additionals
from workspace.crm_analytics.cliente_silver
limit 10

first_name,last_name,email,street,number,additionals
Marta,Jesus,nao_informado,nao_informado,0.0,Conjunto 16
Luana,Almeida,nao_informado,Avenida 56 do Estado Rio Grande do Sul,989.0,Conjunto 17
Frida,Mendes,frida@meu_email.com,Avenida 59 do Estado São Paulo,534.0,sem_complemento
Daniela,Avelino,daniela@exemplo.com,nao_informado,0.0,sem_complemento
Romário,Teixeira,nao_informado,Praça 56 do Estado Bahia,191.0,Apto 12
Marcelo,Barroso,nao_informado,Rua 28 do Estado Rio Grande do Sul,805.0,Conjunto 13
Cristiano,Elísio,cristiano@exemplo.com,Rua 78 do Estado Goiás,877.0,Apto 14
Everton,Barbosa,everton@meu_email.com,Avenida 86 do Estado Distrito Federal,864.0,Apto 14
Gabriela,Alves,gabriela@exemplo.com,nao_informado,0.0,sem_complemento
Luan,Dias,luan@exemplo.com,Avenida 54 do Estado Distrito Federal,889.0,Conjunto 14


**Will, a base de clientes limpa (Silver) ficou excelente. Na semana que vem, vamos lançar uma grande campanha de anúncios regionais e o orçamento é limitado.
Preciso que você crie uma tabela consolidada que me mostre quantos clientes nós temos em cada estado. E para facilitar a minha vida no dashboard, essa tabela já tem que vir ordenada, mostrando os estados com mais clientes no topo.**

1. Criar uma tabela que mostre quantidade de clientes por estado.
2. Agrupar os dados da coluna state
3. Ordenar por do estado maior para o menor.
4. Criar a tabela cliente_gold, dentro do crm_anaytics.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS WORKSPACE.crm_analytics.clientes_estado_gold

In [0]:
%sql
--drop table WORKSPACE.crm_analytics.clientes_estado_gold

In [0]:
from pyspark.sql.functions import desc, col

df_clientes_gold = (spark.read.table('workspace.crm_analytics.cliente_silver')
    .groupBy("state").count()
    .orderBy(col("count").desc()))
df_clientes_gold.write.mode('overwrite').saveAsTable('workspace.crm_analytics.clientes_estado_gold')


In [0]:
%sql
select state, count
from workspace.crm_analytics.clientes_estado_gold

state,count
Pernambuco,8
Rio Grande do Sul,6
Bahia,6
Maranhão,6
Ceará,5
Goiás,5
Mato Grosso,5
Rio Grande do Norte,5
Minas Gerais,4
São Paulo,4
